# Long-Short Volatility Thresholds

Uses rolling volatility thresholds to assign long or short exposure.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
================================================================================
THRESHOLD REGIME TRADING STRATEGY - VERSION 2.1 LONG/SHORT VARIANT
================================================================================

STRATEGY OVERVIEW:
-----------------
Uses volatility-based threshold regime detection to dynamically allocate
between LONG and SHORT positions.

MODIFICATION FROM ORIGINAL:
--------------------------
- HIGH VOLATILITY REGIME: Goes SHORT (-100%) instead of CASH (0%)
- This is a more aggressive strategy that profits from both directions
- Margin interest is charged on short positions

REGIME ALLOCATIONS:
- Regime 0 (Low Vol):  +100% LONG (fully invested)
- Regime 1 (High Vol): -100% SHORT (fully short)

METHODOLOGY:
-----------
1. REGIME DETECTION (Threshold Model with Persistence):
   - Calculate Yang-Zhang volatility using OHLC data
   - Use ROLLING quantiles (not global) to define regime thresholds
   - Apply persistence filter: regime must hold N days before signal change
   - Regime 0 (Low Vol): Full LONG exposure
   - Regime 1 (High Vol): Full SHORT exposure

2. TEMPORAL INTEGRITY (Zero Look-Ahead Bias):
   - Signal generated at Close[T-1] using only data up to T-1
   - Order executed at Open[T] the next trading day
   - Position sized using Close[T-1] prices and portfolio value
   - Rolling quantile windows use closed='left' to exclude current observation
   - All feature engineering explicitly uses shift(1)

3. WALK-FORWARD VALIDATION WITH INTRA-FOLD OPTIMIZATION:
   - Expanding window approach with periodic model refitting
   - For each fold: optimize parameters on Train, evaluate on Validate
   - Out-of-sample performance tracking for robustness assessment
   - Parameter stability analysis across folds

4. HYBRID P&L CALCULATION:
   - Execution days: Use Open[T]/Open[T-1] - 1 for return calculation
   - Holding days: Use Close[T]/Close[T-1] - 1 for return calculation
   - SHORT POSITIONS: P&L inverted (profit when price drops)

5. POSITION SIZING:
   - RESEARCH MODE (COMPOUNDING_MODE=False): Fixed notional on initial capital
   - PRODUCTION MODE (COMPOUNDING_MODE=True): Geometric compounding
   - Cash reserve tracks unrealized gains/losses in research mode

6. TRANSACTION COSTS (IBKR-like):
   - Commission: 0.0035 per share, min $0.35, max 1% of trade value
   - Spread: 0.02% per side
   - Market Impact: 0.03% per side
   - Margin Interest: Fed Funds + 1.5% for SHORT positions (daily accrual)
   - Short Borrow Fee: Additional cost for borrowing shares

ROBUSTNESS METRICS:
------------------
- Deflated Sharpe Ratio (DSR): Adjusts for multiple testing bias
- Probabilistic Sharpe Ratio (PSR): Statistical significance of Sharpe
- Walk-Forward Efficiency (WFE): OOS/IS performance ratio
- Hurst Exponent: Mean-reversion vs trend persistence
- CDaR (Conditional Drawdown at Risk): Tail risk measure
- PSI (Parameter Stability Index): Sensitivity analysis across folds
- Rolling IC: Predictive power consistency
- Global Robustness Index (GRI): Combined normalized score

DATA SOURCE:
-----------
Yahoo Finance - No synthetic data, strict validation

================================================================================
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import warnings
from scipy import stats
from itertools import product
import json

warnings.filterwarnings("ignore")
TICKER = "SPY"


@dataclass
class StrategyConfig:
    """
    Master configuration class for the Threshold Regime Trading Strategy V2.1.

    LONG/SHORT VARIANT:
    - Low volatility regime: LONG (+100%)
    - High volatility regime: SHORT (-100%)
    """

    ticker: str = TICKER
    benchmark_ticker: str = "URTH"
    start_date: str = "2021-01-01"
    end_date: str = "2025-11-22"
    warmup_buffer_days: int = 700
    vol_lookback: int = 20
    quantile_window: int = 252
    n_regimes: int = 2
    regime_quantile: float = 0.5
    yz_k_numerator: float = 0.34
    yz_k_denominator_constant: float = 1.34
    yz_annualization_factor: int = 252
    persistence_days: int = 3
    use_persistence_filter: bool = True
    initial_capital: float = 100000.0
    compounding_mode: bool = True
    leverage: float = 1.0
    low_vol_allocation: float = 1.0
    high_vol_allocation: float = -1.0
    commission_per_share: float = 0.0035
    commission_min: float = 0.35
    commission_max_pct: float = 0.01
    spread_bps: float = 2.0
    market_impact_bps: float = 3.0
    margin_rate_annual: float = 0.06
    short_borrow_fee_annual: float = 0.005
    min_train_window: int = 504
    validation_window: int = 126
    refit_frequency: int = 126
    step_size: int = 63
    vol_lookback_grid: List[int] = field(default_factory=lambda: [10, 15, 20, 30, 42, 63])
    quantile_window_grid: List[int] = field(default_factory=lambda: [126, 189, 252, 378, 504])
    regime_quantile_grid: List[float] = field(default_factory=lambda: [0.3, 0.4, 0.5, 0.6, 0.7])
    persistence_days_grid: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 5])
    fast_mode: bool = False
    optimization_objective: str = "sharpe"
    max_param_combinations: int = 200
    gri_threshold: float = 0.65
    min_psr: float = 0.95
    min_wfe: float = 0.5
    num_trials: int = 100
    psr_benchmark_sharpe: float = 0.0


class YahooDataFetcher:
    """Yahoo Finance data fetcher."""

    def __init__(self):
        pass

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """Fetch historical OHLCV data for a single ticker."""
        print(f"[DATA] Fetching {ticker} from {start_date} to {end_date}...")
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
        except RuntimeError as e:
            raise ValueError(f"API request failed for {ticker}: {str(e)}")
        if "historical" not in data:
            if "Error Message" in str(data):
                raise ValueError(f"Yahoo Finance error for {ticker}: {data}")
            raise ValueError(f"No historical data found for {ticker}")
        historical = data["historical"]
        if len(historical) == 0:
            raise ValueError(f"Empty historical data for {ticker}")
        df = pd.DataFrame(historical)
        df["date"] = pd.to_datetime(df["date"])
        df.set_index("date", inplace=True)
        df.sort_index(inplace=True)
        column_mapping = {
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "adjClose": "Adj Close",
            "volume": "Volume",
        }
        available_cols = {k: v for k, v in column_mapping.items() if k in df.columns}
        df = df[list(available_cols.keys())].rename(columns=available_cols)
        if "Adj Close" in df.columns and "Close" in df.columns:
            adj_ratio = df["Adj Close"] / df["Close"]
            df["Open"] = df["Open"] * adj_ratio
            df["High"] = df["High"] * adj_ratio
            df["Low"] = df["Low"] * adj_ratio
            df["Close"] = df["Adj Close"]
        price_cols = ["Open", "High", "Low", "Close"]
        missing_pct = df[price_cols].isna().sum() / len(df) * 100
        if missing_pct.max() > 5:
            print(f"[WARNING] High missing data for {ticker}: {missing_pct.to_dict()}")
        print(f"[DATA] Retrieved {len(df)} trading days for {ticker}")
        return df


class YangZhangVolatility:
    """Yang-Zhang (2000) volatility estimator implementation."""

    @staticmethod
    def calculate(df: pd.DataFrame, window: int, config: StrategyConfig) -> pd.Series:
        """Calculate Yang-Zhang volatility using OHLC data."""
        log_open = np.log(df["Open"])
        log_high = np.log(df["High"])
        log_low = np.log(df["Low"])
        log_close = np.log(df["Close"])
        log_close_prev = log_close.shift(1)
        overnight_ret = log_open - log_close_prev
        var_overnight = overnight_ret.rolling(window).var()
        open_close_ret = log_close - log_open
        var_open_close = open_close_ret.rolling(window).var()
        rs_component = (log_high - log_close) * (log_high - log_open) + (log_low - log_close) * (
            log_low - log_open
        )
        var_rs = rs_component.rolling(window).mean()
        k = config.yz_k_numerator / (config.yz_k_denominator_constant + (window + 1) / (window - 1))
        yz_variance = var_overnight + k * var_open_close + (1 - k) * var_rs
        yz_volatility = np.sqrt(yz_variance) * np.sqrt(config.yz_annualization_factor)
        return yz_volatility


class ThresholdRegimeDetector:
    """
    Threshold-based regime detection using rolling quantiles with persistence filter.

    LONG/SHORT VARIANT:
    - Regime 0 (Low Vol): Signal = +1.0 (LONG)
    - Regime 1 (High Vol): Signal = -1.0 (SHORT)
    """

    def __init__(
        self,
        config: StrategyConfig,
        vol_lookback: Optional[int] = None,
        quantile_window: Optional[int] = None,
        regime_quantile: Optional[float] = None,
        persistence_days: Optional[int] = None,
    ):
        self.config = config
        self.vol_lookback = vol_lookback or config.vol_lookback
        self.quantile_window = quantile_window or config.quantile_window
        self.regime_quantile = regime_quantile or config.regime_quantile
        self.persistence_days = persistence_days or config.persistence_days

    def detect_regimes(self, df: pd.DataFrame) -> pd.DataFrame:
        """Detect regimes using threshold model on volatility with persistence filter."""
        df = df.copy()
        df["volatility"] = YangZhangVolatility.calculate(df, self.vol_lookback, self.config)
        df["vol_threshold"] = (
            df["volatility"]
            .rolling(self.quantile_window, closed="left")
            .quantile(self.regime_quantile)
        )
        df["regime_raw"] = (df["volatility"] > df["vol_threshold"]).astype(int)
        if self.config.use_persistence_filter and self.persistence_days > 1:
            df["regime"] = self._apply_persistence_filter(df["regime_raw"], self.persistence_days)
        else:
            df["regime"] = df["regime_raw"]
        regime_names = {0: "Low Vol (LONG)", 1: "High Vol (SHORT)"}
        df["regime_name"] = df["regime"].map(regime_names)
        df["regime_duration"] = self._calculate_regime_duration(df["regime"])
        df["returns"] = df["Close"].pct_change()
        df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))
        return df

    def _apply_persistence_filter(self, raw_regime: pd.Series, persistence_days: int) -> pd.Series:
        """Apply persistence filter to raw regime signals."""
        filtered = raw_regime.copy()
        valid_mask = ~raw_regime.isna()
        valid_indices = raw_regime[valid_mask].index
        if len(valid_indices) < persistence_days:
            return filtered
        current_regime = raw_regime.loc[valid_indices[0]]
        consecutive_count = 1
        for i, idx in enumerate(valid_indices):
            raw_value = raw_regime.loc[idx]
            if pd.isna(raw_value):
                filtered.loc[idx] = np.nan
                continue
            if raw_value == current_regime:
                consecutive_count += 1
                filtered.loc[idx] = current_regime
            else:
                new_regime_count = 1
                for j in range(i + 1, min(i + persistence_days, len(valid_indices))):
                    future_idx = valid_indices[j]
                    if raw_regime.loc[future_idx] == raw_value:
                        new_regime_count += 1
                    else:
                        break
                past_count = 0
                for j in range(max(0, i - persistence_days + 1), i):
                    past_idx = valid_indices[j]
                    if raw_regime.loc[past_idx] == raw_value:
                        past_count += 1
                if past_count >= persistence_days - 1:
                    current_regime = raw_value
                    consecutive_count = 1
                    filtered.loc[idx] = current_regime
                else:
                    filtered.loc[idx] = current_regime
        return filtered

    def _calculate_regime_duration(self, regime: pd.Series) -> pd.Series:
        """Calculate how many days the current regime has persisted."""
        duration = pd.Series(index=regime.index, dtype=float)
        current_count = 0
        prev_regime = None
        for idx, val in regime.items():
            if pd.isna(val):
                duration.loc[idx] = np.nan
                continue
            if val == prev_regime:
                current_count += 1
            else:
                current_count = 1
                prev_regime = val
            duration.loc[idx] = current_count
        return duration

    def get_signal(self, regime: int) -> float:
        """
        Convert regime to target allocation signal.

        LONG/SHORT VARIANT:
        - Regime 0 (Low Vol):  +1.0 (LONG)
        - Regime 1 (High Vol): -1.0 (SHORT)

        Args:
            regime: Integer regime label (0 or 1)

        Returns:
            Target allocation: +1.0 (long) or -1.0 (short)
        """
        if pd.isna(regime):
            return np.nan
        if regime == 0:
            return self.config.low_vol_allocation * self.config.leverage
        else:
            return self.config.high_vol_allocation * self.config.leverage


class TransactionCostCalculator:
    """
    Calculates realistic transaction costs for backtesting.

    INCLUDES SHORT-SPECIFIC COSTS:
    - Margin interest on short position value
    - Stock borrow fee for shorting
    """

    def __init__(self, config: StrategyConfig):
        self.config = config

    def calculate_commission(self, shares: float, price: float) -> float:
        """Calculate IBKR-style commission."""
        shares = abs(shares)
        trade_value = shares * price
        raw_commission = shares * self.config.commission_per_share
        min_commission = self.config.commission_min
        max_commission = trade_value * self.config.commission_max_pct
        return max(min_commission, min(raw_commission, max_commission))

    def calculate_spread_cost(self, shares: float, price: float) -> float:
        """Calculate bid-ask spread cost."""
        trade_value = abs(shares) * price
        return trade_value * (self.config.spread_bps / 10000)

    def calculate_market_impact(self, shares: float, price: float) -> float:
        """Calculate market impact cost."""
        trade_value = abs(shares) * price
        return trade_value * (self.config.market_impact_bps / 10000)

    def calculate_total_cost(self, shares: float, price: float) -> float:
        """Calculate total transaction cost."""
        commission = self.calculate_commission(shares, price)
        spread = self.calculate_spread_cost(shares, price)
        impact = self.calculate_market_impact(shares, price)
        return commission + spread + impact

    def calculate_margin_interest(self, borrowed_value: float, days: int = 1) -> float:
        """
        Calculate margin interest on borrowed funds.

        For SHORT positions, this is interest charged on the short sale proceeds
        that must be maintained as collateral.
        """
        daily_rate = self.config.margin_rate_annual / 252
        return abs(borrowed_value) * daily_rate * days

    def calculate_short_borrow_fee(self, short_value: float, days: int = 1) -> float:
        """
        Calculate stock borrow fee for short positions.

        This is the fee paid to the lender of shares for short selling.
        Varies by stock - using conservative average for liquid stocks.
        """
        daily_rate = self.config.short_borrow_fee_annual / 252
        return abs(short_value) * daily_rate * days


class BacktestEngine:
    """
    Production-grade backtesting engine with LONG/SHORT support.

    LONG/SHORT P&L CALCULATION:
    --------------------------
    - LONG position: P&L = shares * (price_end - price_start)
    - SHORT position: P&L = -shares * (price_end - price_start)
      OR equivalently: P&L = |shares| * (price_start - price_end)

    When short, we PROFIT when price goes DOWN.

    SHORT-SPECIFIC COSTS:
    - Margin interest charged daily on short position value
    - Stock borrow fee charged daily
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.cost_calculator = TransactionCostCalculator(config)

    def run_backtest(
        self, df: pd.DataFrame, signals: pd.Series, return_diagnostics: bool = False
    ) -> pd.DataFrame:
        """
        Execute the backtest with LONG/SHORT positions.

        Signals:
        - +1.0: 100% LONG
        - -1.0: 100% SHORT
        - 0.0: FLAT (no position)
        """
        df = df.copy()
        df["target_allocation"] = signals.shift(1)
        results = {
            "date": [],
            "open_price": [],
            "close_price": [],
            "target_alloc": [],
            "actual_alloc": [],
            "shares": [],
            "position_value": [],
            "cash": [],
            "portfolio_value": [],
            "gross_return": [],
            "transaction_cost": [],
            "margin_interest": [],
            "borrow_fee": [],
            "net_return": [],
            "benchmark_return": [],
            "is_execution_day": [],
            "regime": [],
            "signal_change": [],
            "position_type": [],
        }
        current_cash = self.config.initial_capital
        current_shares = 0.0
        cash_reserve = 0.0
        executed_signal = None
        valid_dates = df.dropna(subset=["target_allocation", "Open", "Close"]).index
        if len(valid_dates) == 0:
            raise ValueError("No valid trading dates after warm-up period")
        previous_close = df.loc[valid_dates[0], "Close"]
        for i, date in enumerate(valid_dates):
            open_price = df.loc[date, "Open"]
            close_price = df.loc[date, "Close"]
            target_alloc = df.loc[date, "target_allocation"]
            regime = df.loc[date, "regime"] if "regime" in df.columns else np.nan
            if executed_signal is None:
                signal_changed = True
            else:
                signal_changed = abs(target_alloc - executed_signal) > 0.01
            is_execution_day = signal_changed
            transaction_cost = 0.0
            margin_interest = 0.0
            borrow_fee = 0.0
            if signal_changed:
                if self.config.compounding_mode:
                    current_portfolio_value = self.config.initial_capital + cash_reserve
                    sizing_capital = current_portfolio_value
                else:
                    sizing_capital = self.config.initial_capital
                target_value = target_alloc * sizing_capital * self.config.leverage
                new_shares = target_value / open_price if open_price > 0 else 0
                shares_change = new_shares - current_shares
                if abs(shares_change) > 0.001:
                    transaction_cost = self.cost_calculator.calculate_total_cost(
                        shares_change, open_price
                    )
                if i > 0 and current_shares != 0:
                    overnight_return = current_shares * (open_price - previous_close)
                else:
                    overnight_return = 0.0
                intraday_return = new_shares * (close_price - open_price)
                position_return = overnight_return + intraday_return
                current_shares = new_shares
                executed_signal = target_alloc
            elif current_shares != 0:
                position_return = current_shares * (close_price - previous_close)
            else:
                position_return = 0.0
            if current_shares < 0:
                short_value = abs(current_shares * close_price)
                margin_interest = self.cost_calculator.calculate_margin_interest(short_value, 1)
                borrow_fee = self.cost_calculator.calculate_short_borrow_fee(short_value, 1)
            position_value = current_shares * close_price
            gross_return = position_return
            total_costs = transaction_cost + margin_interest + borrow_fee
            net_return = gross_return - total_costs
            cash_reserve += net_return
            current_portfolio_value = self.config.initial_capital + cash_reserve
            current_cash = current_portfolio_value - position_value
            if previous_close > 0:
                benchmark_return = close_price / previous_close - 1
            else:
                benchmark_return = 0.0
            if current_shares > 0.001:
                position_type = "LONG"
            elif current_shares < -0.001:
                position_type = "SHORT"
            else:
                position_type = "FLAT"
            results["date"].append(date)
            results["open_price"].append(open_price)
            results["close_price"].append(close_price)
            results["target_alloc"].append(target_alloc)
            results["actual_alloc"].append(
                position_value / current_portfolio_value if current_portfolio_value > 0 else 0
            )
            results["shares"].append(current_shares)
            results["position_value"].append(position_value)
            results["cash"].append(current_cash)
            results["portfolio_value"].append(current_portfolio_value)
            results["gross_return"].append(gross_return)
            results["transaction_cost"].append(transaction_cost)
            results["margin_interest"].append(margin_interest)
            results["borrow_fee"].append(borrow_fee)
            results["net_return"].append(net_return)
            results["benchmark_return"].append(benchmark_return)
            results["is_execution_day"].append(is_execution_day)
            results["regime"].append(regime)
            results["signal_change"].append(signal_changed)
            results["position_type"].append(position_type)
            previous_close = close_price
        results_df = pd.DataFrame(results)
        results_df.set_index("date", inplace=True)
        results_df["cumulative_return"] = (
            results_df["portfolio_value"] / self.config.initial_capital - 1
        )
        results_df["cumulative_benchmark"] = (1 + results_df["benchmark_return"]).cumprod() - 1
        results_df["strategy_return"] = results_df["portfolio_value"].pct_change()
        results_df["total_short_costs"] = results_df["margin_interest"] + results_df["borrow_fee"]
        return results_df


class PerformanceAnalytics:
    """Comprehensive performance analysis and robustness testing."""

    def __init__(self, config: StrategyConfig):
        self.config = config

    def calculate_basic_metrics(
        self,
        returns: pd.Series,
        benchmark_returns: Optional[pd.Series] = None,
        annualization_factor: int = 252,
    ) -> Dict[str, float]:
        """Calculate basic performance metrics."""
        returns = returns.dropna()
        if len(returns) < 20:
            return {
                "error": "Insufficient data",
                "total_return": 0,
                "annual_return": 0,
                "volatility": 0,
                "sharpe_ratio": 0,
                "sortino_ratio": 0,
                "max_drawdown": 0,
                "calmar_ratio": 0,
                "num_observations": len(returns),
            }
        total_return = (1 + returns).prod() - 1
        n_years = len(returns) / annualization_factor
        annual_return = (1 + total_return) ** (1 / n_years) - 1 if n_years > 0 else 0
        volatility = returns.std() * np.sqrt(annualization_factor)
        sharpe = annual_return / volatility if volatility > 0 else 0
        downside_returns = returns[returns < 0]
        downside_deviation = (
            downside_returns.std() * np.sqrt(annualization_factor)
            if len(downside_returns) > 0
            else 0
        )
        sortino = annual_return / downside_deviation if downside_deviation > 0 else 0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.cummax()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()
        calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
        var_95 = returns.quantile(0.05)
        cvar_95 = (
            returns[returns <= var_95].mean() if len(returns[returns <= var_95]) > 0 else var_95
        )
        win_rate = (returns > 0).sum() / len(returns)
        skewness = returns.skew()
        kurtosis = returns.kurtosis() + 3
        metrics = {
            "total_return": total_return,
            "annual_return": annual_return,
            "volatility": volatility,
            "sharpe_ratio": sharpe,
            "sortino_ratio": sortino,
            "max_drawdown": max_drawdown,
            "calmar_ratio": calmar,
            "var_95": var_95,
            "cvar_95": cvar_95,
            "win_rate": win_rate,
            "skewness": skewness,
            "kurtosis": kurtosis,
            "num_observations": len(returns),
        }
        if benchmark_returns is not None:
            benchmark_returns = benchmark_returns.reindex(returns.index).dropna()
            if len(benchmark_returns) > 20:
                benchmark_total = (1 + benchmark_returns).prod() - 1
                excess_return = total_return - benchmark_total
                active_returns = returns - benchmark_returns.reindex(returns.index).fillna(0)
                tracking_error = active_returns.std() * np.sqrt(annualization_factor)
                benchmark_n_years = len(benchmark_returns) / annualization_factor
                benchmark_annual = (
                    (1 + benchmark_total) ** (1 / benchmark_n_years) - 1
                    if benchmark_n_years > 0
                    else 0
                )
                info_ratio = (
                    (annual_return - benchmark_annual) / tracking_error if tracking_error > 0 else 0
                )
                metrics["benchmark_return"] = benchmark_total
                metrics["excess_return"] = excess_return
                metrics["tracking_error"] = tracking_error
                metrics["information_ratio"] = info_ratio
        return metrics

    def calculate_deflated_sharpe_ratio(
        self,
        sharpe: float,
        num_observations: int,
        num_trials: int,
        skewness: float = 0,
        kurtosis: float = 3,
    ) -> float:
        """Calculate Deflated Sharpe Ratio (DSR)."""
        if num_trials <= 1:
            return self.calculate_probabilistic_sharpe_ratio(
                sharpe, num_observations, 0, skewness, kurtosis
            )
        try:
            if num_trials > 1:
                euler_mascheroni = 0.5772156649
                z_arg = 1 - 1 / num_trials
                z_arg = max(0.0001, min(0.9999, z_arg))
                z_quantile = stats.norm.ppf(z_arg)
                e_max_arg = 1 - 1 / (num_trials * np.e)
                e_max_arg = max(0.0001, min(0.9999, e_max_arg))
                expected_max_sr = (1 - euler_mascheroni) * stats.norm.ppf(
                    z_arg
                ) + euler_mascheroni * stats.norm.ppf(e_max_arg)
                sr_std = np.sqrt(
                    (1 + 0.5 * sharpe**2 - skewness * sharpe + (kurtosis - 3) / 4 * sharpe**2)
                    / num_observations
                )
                expected_max_sr = expected_max_sr * sr_std
            else:
                expected_max_sr = 0
                sr_std = np.sqrt((1 + 0.5 * sharpe**2) / num_observations)
            if sr_std > 0:
                z_score = (sharpe - expected_max_sr) / sr_std
                dsr = stats.norm.cdf(z_score)
            else:
                dsr = 0.5
            dsr = max(0, min(1, dsr))
        except Exception as e:
            print(f"[WARNING] DSR calculation error: {e}")
            dsr = 0.5
        return dsr

    def calculate_probabilistic_sharpe_ratio(
        self,
        sharpe: float,
        num_observations: int,
        benchmark_sharpe: float = 0,
        skewness: float = 0,
        kurtosis: float = 3,
    ) -> float:
        """Calculate Probabilistic Sharpe Ratio (PSR)."""
        if num_observations < 2:
            return 0.5
        variance_term = 1 + 0.5 * sharpe**2 - skewness * sharpe + (kurtosis - 3) / 4 * sharpe**2
        if variance_term <= 0:
            variance_term = 1
        sharpe_std_error = np.sqrt(variance_term / num_observations)
        if sharpe_std_error > 0:
            z_score = (sharpe - benchmark_sharpe) / sharpe_std_error
            psr = stats.norm.cdf(z_score)
        else:
            psr = 0.5
        return max(0, min(1, psr))

    def calculate_hurst_exponent(self, returns: pd.Series, max_lag: int = 100) -> float:
        """Calculate Hurst exponent using R/S analysis."""
        returns = returns.dropna().values
        n = len(returns)
        if n < 50:
            return 0.5
        max_lag = min(max_lag, n // 4)
        lags = range(10, max_lag)
        rs_values = []
        for lag in lags:
            num_chunks = n // lag
            rs_list = []
            for i in range(num_chunks):
                chunk = returns[i * lag : (i + 1) * lag]
                mean_chunk = chunk.mean()
                cum_dev = np.cumsum(chunk - mean_chunk)
                r = cum_dev.max() - cum_dev.min()
                s = chunk.std()
                if s > 0:
                    rs_list.append(r / s)
            if len(rs_list) > 0:
                rs_values.append((np.log(lag), np.log(np.mean(rs_list))))
        if len(rs_values) < 2:
            return 0.5
        rs_array = np.array(rs_values)
        slope, _, _, _, _ = stats.linregress(rs_array[:, 0], rs_array[:, 1])
        return max(0, min(1, slope))

    def calculate_cdar(self, returns: pd.Series, alpha: float = 0.05) -> float:
        """Calculate Conditional Drawdown at Risk (CDaR)."""
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.cummax()
        drawdowns = (cumulative - running_max) / running_max
        threshold = drawdowns.quantile(alpha)
        worst_drawdowns = drawdowns[drawdowns <= threshold]
        if len(worst_drawdowns) > 0:
            return worst_drawdowns.mean()
        else:
            return drawdowns.min()

    def calculate_rolling_ic(
        self, signals: pd.Series, forward_returns: pd.Series, window: int = 63
    ) -> pd.Series:
        """Calculate rolling Information Coefficient (IC)."""
        aligned_signals = signals.shift(1)
        rolling_ic = aligned_signals.rolling(window).corr(forward_returns)
        return rolling_ic

    def calculate_parameter_stability_index(self, sharpes_by_fold: List[float]) -> float:
        """Calculate Parameter Stability Index (PSI)."""
        if len(sharpes_by_fold) < 2:
            return 1.0
        sharpes = np.array([s for s in sharpes_by_fold if not np.isnan(s) and s != 0])
        if len(sharpes) < 2:
            return 0.5
        mean_sharpe = np.mean(sharpes)
        std_sharpe = np.std(sharpes)
        if mean_sharpe != 0:
            cv = abs(std_sharpe / mean_sharpe)
            psi = 1 / (1 + cv)
        else:
            psi = 0.5 if std_sharpe == 0 else 0.3
        return psi

    def calculate_global_robustness_index(self, metrics: Dict[str, float]) -> float:
        """Calculate Global Robustness Index (GRI)."""
        weights = {"dsr": 0.2, "psr": 0.2, "wfe": 0.2, "hurst": 0.1, "cdar": 0.15, "psi": 0.15}
        normalized = {}
        normalized["dsr"] = max(0, min(1, metrics.get("dsr", 0.5)))
        normalized["psr"] = max(0, min(1, metrics.get("psr", 0.5)))
        wfe = metrics.get("wfe", 0.5)
        normalized["wfe"] = max(0, min(1, wfe / 1.5))
        normalized["hurst"] = max(0, min(1, metrics.get("hurst", 0.5)))
        cdar = metrics.get("cdar", -0.2)
        normalized["cdar"] = max(0, min(1, 1 + cdar * 2))
        normalized["psi"] = max(0, min(1, metrics.get("psi", 0.5)))
        gri = sum((weights[k] * normalized[k] for k in weights))
        return gri


class HyperparameterOptimizer:
    """Intra-fold hyperparameter optimization for walk-forward validation."""

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.analytics = PerformanceAnalytics(config)

    def generate_param_grid(self) -> List[Dict[str, Any]]:
        """Generate parameter combinations for grid search."""
        if self.config.fast_mode:
            vol_lookbacks = [20, 40]
            quantile_windows = [252]
            regime_quantiles = [0.5]
            persistence_days_list = [3]
            print("    [FAST MODE] Using reduced parameter grid")
        else:
            vol_lookbacks = self.config.vol_lookback_grid
            quantile_windows = self.config.quantile_window_grid
            regime_quantiles = self.config.regime_quantile_grid
            persistence_days_list = self.config.persistence_days_grid
        all_params = []
        for vl, qw, rq, pd in product(
            vol_lookbacks, quantile_windows, regime_quantiles, persistence_days_list
        ):
            if qw >= vl:
                all_params.append(
                    {
                        "vol_lookback": vl,
                        "quantile_window": qw,
                        "regime_quantile": rq,
                        "persistence_days": pd,
                    }
                )
        if len(all_params) > self.config.max_param_combinations:
            np.random.seed(42)
            indices = np.random.choice(
                len(all_params), self.config.max_param_combinations, replace=False
            )
            all_params = [all_params[i] for i in indices]
        return all_params

    def evaluate_params(
        self, params: Dict[str, Any], df: pd.DataFrame, config: StrategyConfig
    ) -> Dict[str, float]:
        """Evaluate a single parameter combination on training data."""
        try:
            detector = ThresholdRegimeDetector(
                config,
                vol_lookback=params["vol_lookback"],
                quantile_window=params["quantile_window"],
                regime_quantile=params["regime_quantile"],
                persistence_days=params["persistence_days"],
            )
            df_with_regimes = detector.detect_regimes(df)
            valid_regimes = df_with_regimes["regime"].dropna()
            if len(valid_regimes) < 60:
                return {"sharpe_ratio": -999, "valid": False}
            signals = df_with_regimes["regime"].apply(detector.get_signal)
            backtest_engine = BacktestEngine(config)
            results = backtest_engine.run_backtest(df_with_regimes, signals)
            returns = results["strategy_return"].dropna()
            if len(returns) < 30:
                return {"sharpe_ratio": -999, "valid": False}
            metrics = self.analytics.calculate_basic_metrics(returns)
            metrics["valid"] = True
            metrics["num_trades"] = results["is_execution_day"].sum()
            return metrics
        except Exception as e:
            return {"sharpe_ratio": -999, "valid": False, "error": str(e)}

    def optimize(
        self, df: pd.DataFrame, verbose: bool = False
    ) -> Tuple[Dict[str, Any], Dict[str, float]]:
        """Find optimal parameters using grid search."""
        param_grid = self.generate_param_grid()
        if verbose:
            print(f"    Testing {len(param_grid)} parameter combinations...")
        best_params = None
        best_metrics = {"sharpe_ratio": -999}
        best_objective = -np.inf
        for i, params in enumerate(param_grid):
            metrics = self.evaluate_params(params, df, self.config)
            if not metrics.get("valid", False):
                continue
            if self.config.optimization_objective == "sharpe":
                objective = metrics.get("sharpe_ratio", -999)
            elif self.config.optimization_objective == "sortino":
                objective = metrics.get("sortino_ratio", -999)
            elif self.config.optimization_objective == "calmar":
                objective = metrics.get("calmar_ratio", -999)
            elif self.config.optimization_objective == "return_dd_ratio":
                ret = metrics.get("total_return", 0)
                dd = abs(metrics.get("max_drawdown", -1))
                objective = ret / dd if dd > 0 else -999
            else:
                objective = metrics.get("sharpe_ratio", -999)
            if objective > best_objective:
                best_objective = objective
                best_params = params
                best_metrics = metrics
        if best_params is None:
            best_params = {
                "vol_lookback": self.config.vol_lookback,
                "quantile_window": self.config.quantile_window,
                "regime_quantile": self.config.regime_quantile,
                "persistence_days": self.config.persistence_days,
            }
        if verbose:
            print(
                f"    Best params: vol_lb={best_params['vol_lookback']}, q_win={best_params['quantile_window']}, q={best_params['regime_quantile']}, pers={best_params['persistence_days']}"
            )
            print(
                f"    Best objective ({self.config.optimization_objective}): {best_objective:.3f}"
            )
        return (best_params, best_metrics)


class WalkForwardValidator:
    """Walk-Forward Validation with expanding windows and intra-fold optimization."""

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.analytics = PerformanceAnalytics(config)
        self.optimizer = HyperparameterOptimizer(config)

    def run_walk_forward(self, df: pd.DataFrame, verbose: bool = True) -> Dict[str, Any]:
        """Execute walk-forward validation with intra-fold optimization."""
        default_detector = ThresholdRegimeDetector(self.config)
        df_initial = default_detector.detect_regimes(df)
        max_warmup = max(self.config.quantile_window_grid) + max(self.config.vol_lookback_grid)
        valid_indices = df_initial.dropna(subset=["regime", "vol_threshold"]).index
        min_required = self.config.min_train_window + self.config.validation_window + max_warmup
        if len(valid_indices) < min_required:
            raise ValueError(
                f"Insufficient data after warm-up. Need {min_required}, have {len(valid_indices)}"
            )
        fold_results = []
        oos_returns_list = []
        param_history = []
        num_trials = len(self.optimizer.generate_param_grid())
        fold_num = 0
        train_end_idx = self.config.min_train_window
        if verbose:
            print("\n" + "=" * 80)
            print("WALK-FORWARD VALIDATION - LONG/SHORT VARIANT")
            print("=" * 80)
            print(f"Total observations: {len(valid_indices)}")
            print(f"Min training window: {self.config.min_train_window}")
            print(f"Validation window: {self.config.validation_window}")
            print(f"Step size: {self.config.step_size}")
            print(f"Parameter combinations per fold: {num_trials}")
            print(f"Regime 0 (Low Vol): LONG (+{self.config.low_vol_allocation * 100:.0f}%)")
            print(f"Regime 1 (High Vol): SHORT ({self.config.high_vol_allocation * 100:.0f}%)")
            print("-" * 80)
        while train_end_idx + self.config.validation_window <= len(valid_indices):
            fold_num += 1
            train_indices = valid_indices[:train_end_idx]
            val_indices = valid_indices[
                train_end_idx : train_end_idx + self.config.validation_window
            ]
            train_start_date = train_indices[0]
            train_end_date = train_indices[-1]
            val_start_date = val_indices[0]
            val_end_date = val_indices[-1]
            if verbose:
                print(f"\nFold {fold_num}:")
                print(
                    f"  Train: {train_start_date.date()} to {train_end_date.date()} ({len(train_indices)} days)"
                )
                print(
                    f"  Valid: {val_start_date.date()} to {val_end_date.date()} ({len(val_indices)} days)"
                )
            train_df = df.loc[train_indices]
            optimal_params, is_metrics = self.optimizer.optimize(train_df, verbose=verbose)
            param_history.append({"fold": fold_num, **optimal_params})
            is_sharpe = is_metrics.get("sharpe_ratio", 0)
            is_return = is_metrics.get("total_return", 0)
            is_max_dd = is_metrics.get("max_drawdown", 0)
            is_num_trades = is_metrics.get("num_trades", 0)
            val_df = df.loc[val_indices]
            optimal_detector = ThresholdRegimeDetector(
                self.config,
                vol_lookback=optimal_params["vol_lookback"],
                quantile_window=optimal_params["quantile_window"],
                regime_quantile=optimal_params["regime_quantile"],
                persistence_days=optimal_params["persistence_days"],
            )
            full_val_df = df.loc[:val_end_date]
            df_with_regimes_val = optimal_detector.detect_regimes(full_val_df)
            val_regimes = df_with_regimes_val.loc[val_indices]
            val_signals = val_regimes["regime"].apply(optimal_detector.get_signal)
            backtest_engine = BacktestEngine(self.config)
            val_results = backtest_engine.run_backtest(val_regimes, val_signals)
            val_returns = val_results["strategy_return"].dropna()
            oos_num_trades = val_results["is_execution_day"].sum()
            oos_signal_changes = val_results["signal_change"].sum()
            long_days = (val_results["position_type"] == "LONG").sum()
            short_days = (val_results["position_type"] == "SHORT").sum()
            if len(val_returns) > 10:
                oos_metrics = self.analytics.calculate_basic_metrics(val_returns)
                oos_sharpe = oos_metrics.get("sharpe_ratio", 0)
                oos_return = oos_metrics.get("total_return", 0)
                oos_max_dd = oos_metrics.get("max_drawdown", 0)
            else:
                oos_sharpe = 0
                oos_return = 0
                oos_max_dd = 0
            oos_returns_list.append(val_returns)
            if is_sharpe != 0 and (not np.isnan(is_sharpe)):
                wfe = oos_sharpe / is_sharpe
            else:
                wfe = 0
            fold_result = {
                "fold": fold_num,
                "train_start": train_start_date,
                "train_end": train_end_date,
                "val_start": val_start_date,
                "val_end": val_end_date,
                "train_days": len(train_indices),
                "val_days": len(val_indices),
                "is_return": is_return,
                "is_sharpe": is_sharpe,
                "is_max_dd": is_max_dd,
                "is_num_trades": is_num_trades,
                "oos_return": oos_return,
                "oos_sharpe": oos_sharpe,
                "oos_max_dd": oos_max_dd,
                "oos_num_trades": oos_num_trades,
                "oos_signal_changes": oos_signal_changes,
                "oos_long_days": long_days,
                "oos_short_days": short_days,
                "wfe": wfe,
                **{f"param_{k}": v for k, v in optimal_params.items()},
            }
            fold_results.append(fold_result)
            if verbose:
                print(
                    f"  IS: Sharpe={is_sharpe:.3f}, Ret={is_return * 100:.2f}%, Trades={is_num_trades}"
                )
                print(
                    f"  OOS: Sharpe={oos_sharpe:.3f}, Ret={oos_return * 100:.2f}%, Trades={oos_num_trades}, WFE={wfe:.3f}"
                )
                print(f"  Position: {long_days} LONG days, {short_days} SHORT days")
            train_end_idx += self.config.step_size
        if verbose:
            print("\n" + "-" * 80)
            print("AGGREGATE WALK-FORWARD RESULTS")
            print("-" * 80)
        if len(oos_returns_list) > 0:
            all_oos_returns = pd.concat(oos_returns_list)
        else:
            raise ValueError("No out-of-sample returns generated")
        agg_metrics = self.analytics.calculate_basic_metrics(all_oos_returns)
        valid_wfes = [
            f["wfe"] for f in fold_results if not np.isnan(f["wfe"]) and abs(f["wfe"]) < 10
        ]
        avg_wfe = np.mean(valid_wfes) if len(valid_wfes) > 0 else 0
        skewness = all_oos_returns.skew()
        kurtosis = all_oos_returns.kurtosis() + 3
        dsr = self.analytics.calculate_deflated_sharpe_ratio(
            sharpe=agg_metrics.get("sharpe_ratio", 0),
            num_observations=len(all_oos_returns),
            num_trials=num_trials * len(fold_results),
            skewness=skewness,
            kurtosis=kurtosis,
        )
        psr = self.analytics.calculate_probabilistic_sharpe_ratio(
            sharpe=agg_metrics.get("sharpe_ratio", 0),
            num_observations=len(all_oos_returns),
            benchmark_sharpe=self.config.psr_benchmark_sharpe,
            skewness=skewness,
            kurtosis=kurtosis,
        )
        hurst = self.analytics.calculate_hurst_exponent(all_oos_returns)
        cdar = self.analytics.calculate_cdar(all_oos_returns)
        oos_sharpes = [f["oos_sharpe"] for f in fold_results]
        psi = self.analytics.calculate_parameter_stability_index(oos_sharpes)
        gri_components = {
            "dsr": dsr,
            "psr": psr,
            "wfe": avg_wfe,
            "hurst": hurst,
            "cdar": cdar,
            "psi": psi,
        }
        gri = self.analytics.calculate_global_robustness_index(gri_components)
        total_long_days = sum((f["oos_long_days"] for f in fold_results))
        total_short_days = sum((f["oos_short_days"] for f in fold_results))
        aggregate_results = {
            "num_folds": len(fold_results),
            "total_oos_days": len(all_oos_returns),
            "total_param_trials": num_trials * len(fold_results),
            "oos_total_return": agg_metrics.get("total_return", 0),
            "oos_annual_return": agg_metrics.get("annual_return", 0),
            "oos_volatility": agg_metrics.get("volatility", 0),
            "oos_sharpe": agg_metrics.get("sharpe_ratio", 0),
            "oos_sortino": agg_metrics.get("sortino_ratio", 0),
            "oos_max_drawdown": agg_metrics.get("max_drawdown", 0),
            "oos_calmar": agg_metrics.get("calmar_ratio", 0),
            "avg_wfe": avg_wfe,
            "dsr": dsr,
            "psr": psr,
            "hurst": hurst,
            "cdar": cdar,
            "psi": psi,
            "gri": gri,
            "is_robust": gri > self.config.gri_threshold,
            "total_long_days": total_long_days,
            "total_short_days": total_short_days,
        }
        if verbose:
            print(f"\nTotal Folds: {aggregate_results['num_folds']}")
            print(f"Total Parameter Trials: {aggregate_results['total_param_trials']}")
            print(f"OOS Total Return: {aggregate_results['oos_total_return'] * 100:.2f}%")
            print(f"OOS Annual Return: {aggregate_results['oos_annual_return'] * 100:.2f}%")
            print(f"OOS Sharpe: {aggregate_results['oos_sharpe']:.3f}")
            print(f"OOS Max DD: {aggregate_results['oos_max_drawdown'] * 100:.2f}%")
            print(f"\nPosition Breakdown:")
            print(f"  Total LONG days: {total_long_days}")
            print(f"  Total SHORT days: {total_short_days}")
            print(f"\nRobustness Metrics:")
            print(f"  Avg WFE: {avg_wfe:.3f}")
            print(f"  DSR: {dsr:.3f}")
            print(f"  PSR: {psr:.3f}")
            print(f"  Hurst: {hurst:.3f}")
            print(f"  CDaR: {cdar * 100:.2f}%")
            print(f"  PSI: {psi:.3f}")
            print(f"  GRI: {gri:.3f}")
            print(
                f"\nDeployment Ready: {('YES' if aggregate_results['is_robust'] else 'NO')} (GRI {('>' if aggregate_results['is_robust'] else '<=')} {self.config.gri_threshold})"
            )
        zero_sharpe_folds = [f for f in fold_results if abs(f["oos_sharpe"]) < 0.01]
        if len(zero_sharpe_folds) > 0 and verbose:
            print(f"\n[DIAGNOSTIC] {len(zero_sharpe_folds)} folds with zero OOS Sharpe:")
            for f in zero_sharpe_folds:
                print(
                    f"  Fold {f['fold']}: {f['val_start'].date()} to {f['val_end'].date()}, trades={f['oos_num_trades']}, signals={f['oos_signal_changes']}"
                )
        return {
            "fold_results": fold_results,
            "aggregate_metrics": aggregate_results,
            "oos_returns": all_oos_returns,
            "param_history": param_history,
        }


def main():
    """
    Main execution function for Threshold Regime Trading Strategy V2.1.

    LONG/SHORT VARIANT:
    - Low volatility: LONG (+100%)
    - High volatility: SHORT (-100%)
    """
    print("=" * 80)
    print(" " * 15 + "THRESHOLD REGIME TRADING STRATEGY V2.1")
    print(" " * 10 + f"{TICKER} - LONG/SHORT VARIANT")
    print("=" * 80)
    print(f"[INFO] Trading Ticker: {TICKER}")
    print(f"[INFO] Strategy: LONG in low vol, SHORT in high vol")
    print("\n[STEP 1] Initializing Configuration...")
    config = StrategyConfig()
    print(f"  Asset: {config.ticker}")
    print(f"  Benchmark: {config.benchmark_ticker}")
    print(f"  Period: {config.start_date} to {config.end_date}")
    print(f"  Initial Capital: ${config.initial_capital:,.0f}")
    print(f"  Compounding Mode: {('Production' if config.compounding_mode else 'Research')}")
    print(f"  Persistence Filter: {config.persistence_days} days")
    print(f"  Low Vol Allocation: {config.low_vol_allocation * 100:+.0f}% (LONG)")
    print(f"  High Vol Allocation: {config.high_vol_allocation * 100:+.0f}% (SHORT)")
    print(f"  Margin Rate: {config.margin_rate_annual * 100:.1f}%")
    print(f"  Short Borrow Fee: {config.short_borrow_fee_annual * 100:.2f}%")
    print("\n[STEP 2] Fetching Market Data from Yahoo Finance...")
    start_dt = datetime.strptime(config.start_date, "%Y-%m-%d")
    extended_start = start_dt - timedelta(days=config.warmup_buffer_days)
    extended_start_str = extended_start.strftime("%Y-%m-%d")
    print(f"  Extended start for warm-up: {extended_start_str}")
    fetcher = YahooDataFetcher()
    try:
        df = fetcher.fetch_historical_data(config.ticker, extended_start_str, config.end_date)
    except ValueError as e:
        print(f"\n[ERROR] Failed to fetch data: {str(e)}")
        print("Please verify:")
        print("  1. Yahoo Finance key is valid")
        print("  2. Ticker symbol is correct")
        print("  3. Network connectivity")
        return None
    print(f"  Data range: {df.index.min().date()} to {df.index.max().date()}")
    print(f"  Total observations: {len(df)}")
    print(f"\n  Fetching benchmark data ({config.benchmark_ticker})...")
    try:
        benchmark_df = fetcher.fetch_historical_data(
            config.benchmark_ticker, extended_start_str, config.end_date
        )
        benchmark_returns = benchmark_df["Close"].pct_change()
    except ValueError as e:
        print(f"  [WARNING] Could not fetch benchmark: {e}")
        benchmark_returns = None
    print("\n[STEP 3] Running Walk-Forward Validation...")
    wf_validator = WalkForwardValidator(config)
    wf_results = wf_validator.run_walk_forward(df, verbose=True)
    print("\n[STEP 4] Running Full Backtest with Optimal Parameters...")
    param_history = wf_results["param_history"]
    final_params = (
        param_history[-1]
        if len(param_history) > 0
        else {
            "vol_lookback": config.vol_lookback,
            "quantile_window": config.quantile_window,
            "regime_quantile": config.regime_quantile,
            "persistence_days": config.persistence_days,
        }
    )
    print(f"  Using parameters from final fold:")
    print(f"    vol_lookback: {final_params.get('vol_lookback', config.vol_lookback)}")
    print(f"    quantile_window: {final_params.get('quantile_window', config.quantile_window)}")
    print(f"    regime_quantile: {final_params.get('regime_quantile', config.regime_quantile)}")
    print(f"    persistence_days: {final_params.get('persistence_days', config.persistence_days)}")
    final_detector = ThresholdRegimeDetector(
        config,
        vol_lookback=final_params.get("vol_lookback", config.vol_lookback),
        quantile_window=final_params.get("quantile_window", config.quantile_window),
        regime_quantile=final_params.get("regime_quantile", config.regime_quantile),
        persistence_days=final_params.get("persistence_days", config.persistence_days),
    )
    df_with_regimes = final_detector.detect_regimes(df)
    backtest_start = pd.to_datetime(config.start_date)
    backtest_end = pd.to_datetime(config.end_date)
    signals = df_with_regimes["regime"].apply(final_detector.get_signal)
    backtest_engine = BacktestEngine(config)
    backtest_results = backtest_engine.run_backtest(df_with_regimes, signals)
    backtest_results = backtest_results[
        (backtest_results.index >= backtest_start) & (backtest_results.index <= backtest_end)
    ]
    final_value = backtest_results["portfolio_value"].iloc[-1]
    total_return = (final_value / config.initial_capital - 1) * 100
    total_costs = backtest_results["transaction_cost"].sum()
    total_margin_interest = backtest_results["margin_interest"].sum()
    total_borrow_fees = backtest_results["borrow_fee"].sum()
    num_trades = backtest_results["is_execution_day"].sum()
    long_days = (backtest_results["position_type"] == "LONG").sum()
    short_days = (backtest_results["position_type"] == "SHORT").sum()
    flat_days = (backtest_results["position_type"] == "FLAT").sum()
    print(f"\n  Final Portfolio Value: ${final_value:,.2f}")
    print(f"  Total Return: {total_return:.2f}%")
    print(f"  Number of Trades: {num_trades}")
    print(f"\n  Position Breakdown:")
    print(f"    LONG days:  {long_days}")
    print(f"    SHORT days: {short_days}")
    print(f"    FLAT days:  {flat_days}")
    print(f"\n  Cost Breakdown:")
    print(f"    Transaction Costs: ${total_costs:,.2f}")
    print(f"    Margin Interest:   ${total_margin_interest:,.2f}")
    print(f"    Borrow Fees:       ${total_borrow_fees:,.2f}")
    print(f"    Total Costs:       ${total_costs + total_margin_interest + total_borrow_fees:,.2f}")
    print("\n[STEP 5] Exporting Results...")
    output_base = f"results/long-short-volatility-thresholds/{TICKER.lower()}_longshort"
    backtest_results.to_csv(f"{output_base}_backtest.csv")
    print(f"  Saved: threshold_regime_{TICKER.lower()}_longshort_backtest.csv")
    wf_df = pd.DataFrame(wf_results["fold_results"])
    wf_df.to_csv(f"{output_base}_walkforward.csv", index=False)
    print(f"  Saved: threshold_regime_{TICKER.lower()}_longshort_walkforward.csv")
    param_df = pd.DataFrame(wf_results["param_history"])
    param_df.to_csv(f"{output_base}_params.csv", index=False)
    print(f"  Saved: threshold_regime_{TICKER.lower()}_longshort_params.csv")
    df_with_regimes.to_csv(f"{output_base}_data.csv")
    print(f"  Saved: threshold_regime_{TICKER.lower()}_longshort_data.csv")
    agg_metrics = wf_results["aggregate_metrics"]
    with open(f"{output_base}_metrics.json", "w") as f:
        json.dump(agg_metrics, f, indent=2, default=str)
    print(f"  Saved: threshold_regime_{TICKER.lower()}_longshort_metrics.json")
    print("\n" + "=" * 80)
    print(f"EXECUTION COMPLETE - V2.1 {TICKER} LONG/SHORT VARIANT")
    print("=" * 80)
    agg = wf_results["aggregate_metrics"]
    print(f"\nStrategy: Threshold Regime Trading V2.1 LONG/SHORT on {config.ticker}")
    print(f"Period: {backtest_results.index.min().date()} to {backtest_results.index.max().date()}")
    print(f"\nKey Results:")
    print(f"  Total Return: {total_return:.2f}%")
    print(f"  OOS Sharpe (Walk-Forward): {agg['oos_sharpe']:.3f}")
    print(f"  Number of Trades: {num_trades}")
    print(f"  Global Robustness Index: {agg['gri']:.3f}")
    print(f"\nLONG/SHORT Variant:")
    print(f"  - Regime 0 (Low Vol):  LONG (+{config.low_vol_allocation * 100:.0f}%)")
    print(f"  - Regime 1 (High Vol): SHORT ({config.high_vol_allocation * 100:.0f}%)")
    print(f"  - Total LONG days: {long_days}")
    print(f"  - Total SHORT days: {short_days}")
    print(
        f"\nDeployment Recommendation: {('PROCEED' if agg['is_robust'] else 'REQUIRES FURTHER ANALYSIS')}"
    )
    if not agg["is_robust"]:
        print(f"\n  GRI ({agg['gri']:.3f}) is below threshold ({config.gri_threshold})")
        print("  Consider:")
        print("    - Adjusting parameter search space")
        print("    - Testing alternative regime detection methods")
        print("    - Extending training window")
        print("    - Adding additional signal filters")
    return {
        "config": config,
        "data": df_with_regimes,
        "backtest_results": backtest_results,
        "wf_results": wf_results,
    }


if __name__ == "__main__":
    results = main()
